# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Tim Schmale | |
| Member|Lennart Oberkönig | |
| Member *(optional)* | | |


## 2. Mission Title & Research Question

**Title:** *What Drives Negative Price Events in the German-Luxembourg Day-Ahead Electricity Market?*

**Research question:**  
Under which market conditions do negative day-ahead electricity prices occur in the German-Luxembourg bidding zone, and how strongly are these events associated with external factors such as weather and seasonality?

**Why it matters:**  

European electricity markets have changed substantially due to the increasing integration of renewable energy sources. Especially in Germany, wind and solar generation play a central role in wholesale price formation, as they have very low marginal costs and are prioritized in market dispatch. This can be described as the merit-order effect, which is based on the idea that the demand for electricity is quasi inelastic and that renewable feed-in can therefore reduce prices by replacing conventional generation (Gürtler und Paulsen, 2018, P. 150).

In this project, we focus on negative day-ahead prices in the German-Luxembourg electricity bidding zone. Negative prices are economically relevant because they indicate periods of excess supply, limited flexibility and changing market incentives (Seel et al., 2021, P. 3). Understanding the market conditions under which these events occur is important for supply and demand management. Thus, we investigate how negative price events are associated with renewable generation, weather-driven supply conditions, and seasonality.

In order to forecast, if the day-ahead price of the next hour will be negative, we base our project on an hourly-based time-series dataset given by Neon Neue Energieökonomik, Technical University of Berlin, ETH Zürich and DIW Berlin. Using this dataset, we begin with data investigation and preparation, enriching the power system data with weather information. After that, we investigate the underlying causal stuctures of the data. Next, we apply k-means clustering as an unsupervised learning approach to identify different market situations. Based on these insides, we build a random forest classifier as our baseline model. Finally, we challenge these results with a neural network as a state of the art modelling approach.

## 3. Data

**Source(s):**  
Open Power System Data
Open Power System Data is provided by Neon Neue Energieökonomik, Technical University of Berlin, ETH Zürich and DIW Berlin - Wiese et al. 2019 
https://data.open-power-system-data.org/time_series/
Weather Data
Weather is provided by Neon Neue Energieökonomik, Technical University of Berlin, ETH Zürich and DIW Berlin - Wiese et al. 2019 
https://data.open-power-system-data.org/weather_data/


**Unit of observation:** One row represents one hour in the German-Luxembourg electricity bidding zone.

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
|cet_cest_timestamp|datetime|Identifier | Start of timeperiod in Central European (Summer-) Time|
|DE_LU_price_day_ahead|numeric| target | Day-ahead spot price for DE-LU (bidding zone) in EUR|
|DE_LU_solar_generation_actual|numeric| feature | Actual solar generation in DE-LU (bidding zone) in MW|
|DE_LU_wind_generation_actual|numeric| feature | Actual wind generation in DE-LU (bidding zone) in MW|
|DE_LU_load_actual_entsoe_transparency|numeric| feature | Total load in DE-LU (bidding zone) in MW as published on ENTSO-E Transparency Platform|
|hour|integer| feature | Hour of the day - derived from cet_cest_timestamp|
|weekday|string| feature | Day of the week - derived from cet_cest_timestamp|
|month|string| feature | Month of the year - derived from cet_cest_timestamp|
|DE_temperature|float| feature | Temperature weather variable for DE in degrees C|
|DE_radiation_direct_horizontal|float| feature | Radiation_direct_horizontal weather variable for DE in W/m2|
|DE_radiation_diffuse_horizontal|float| feature | Radiation_diffuse_horizontal weather variable for DE in W/m2|





**Potential data quality issues:**  
1. Missing data for renewable energy capacity of Luxembourg
2. Missing Values
3. Unequal distribution of positive and negative prices


**Reading, joining and transforming data**

In [22]:
import pandas as pd

from pathlib import Path

BASE = Path().resolve()
DATA = BASE / "data"

# Load OPSD data
opsd = pd.read_csv(DATA / "time_series_60min_singleindex.csv", delimiter=",")

# Load weather data 
weather = pd.read_csv(DATA / "weather_data.csv", delimiter=",")

# reduce the data to the relevant columns
opsd = opsd[[
    "utc_timestamp",
    "cet_cest_timestamp",
    "DE_LU_price_day_ahead",
    "DE_LU_solar_generation_actual",
    "DE_LU_wind_generation_actual",
    "DE_LU_load_actual_entsoe_transparency",
    "DE_wind_capacity",
    "DE_solar_capacity"]]

weather = weather[[
    "utc_timestamp",
    "DE_temperature",
    "DE_radiation_direct_horizontal",
    "DE_radiation_diffuse_horizontal"]]

# merge the two dataframes on the timestamp
df = pd.merge(opsd, weather, on="utc_timestamp", how="inner")

# filter the data to the relevant time period and convert the timestamp to the correct timezone
df = df[df["cet_cest_timestamp"]>="2018-09-30T23:00:00Z"]
df["cet_cest_timestamp"] = pd.to_datetime(df["cet_cest_timestamp"], utc=True).dt.tz_convert("Europe/Berlin")
df = df.set_index("cet_cest_timestamp").drop(columns=["utc_timestamp"])
df

,DE_LU_price_day_ahead,DE_LU_solar_generation_actual,DE_LU_wind_generation_actual,DE_LU_load_actual_entsoe_transparency,DE_wind_capacity,DE_solar_capacity,DE_temperature,DE_radiation_direct_horizontal,DE_radiation_diffuse_horizontal
cet_cest_timestamp,,,,,,,,,
2018-10-01 00:00:00+02:00,NaN,NaN,5932.0,NaN,47730.0,46099.0,8.676,0.0,0.0
2018-10-01 01:00:00+02:00,56.10,NaN,6042.0,NaN,47730.0,46099.0,8.258,0.0,0.0
2018-10-01 02:00:00+02:00,51.41,NaN,6021.0,41874.0,47730.0,46099.0,7.889,0.0,0.0
2018-10-01 03:00:00+02:00,47.38,NaN,6342.0,42713.0,47730.0,46099.0,7.620,0.0,0.0
2018-10-01 04:00:00+02:00,47.59,NaN,7144.0,44165.0,47730.0,46099.0,7.395,0.0,0.0
...,...,...,...,...,...,...,...,...,...
2019-12-31 20:00:00+01:00,42.20,0.0,8875.0,47928.0,NaN,NaN,0.767,0.0,0.0
2019-12-31 21:00:00+01:00,39.74,0.0,7652.0,46235.0,NaN,NaN,0.656,0.0,0.0
2019-12-31 22:00:00+01:00,38.88,0.0,7283.0,45871.0,NaN,NaN,0.476,0.0,0.0


**Addressing the Data Quality Issues:**


**1. Missing data for renewable energy capacity of Luxembourg**

The data of Luxembourgs renewable energy capacity (solar and wind) is not available. To be able to capture the growth of the renewable energy capacity, we use the German capacity as a proxy for the German-Luxembourg electricity bidding zone.

**2. Missing Values**

In [24]:
# check which columns have missing values
df.isna().sum()

DE_LU_price_day_ahead                     3
DE_LU_solar_generation_actual             7
DE_LU_wind_generation_actual              0
DE_LU_load_actual_entsoe_transparency    22
DE_wind_capacity                         25
DE_solar_capacity                        25
DE_temperature                            0
DE_radiation_direct_horizontal            0
DE_radiation_diffuse_horizontal           0
dtype: int64

To address the missing values in the different columns, we use varying appoaches based on the functional form of the column over time. Here, the capacity columns are given as step functions over time, which is why we choose to use the fill the missing values forward. This means the values of the previous hour is assumed for the missing values. In comparison, the other columns are not following such an easy functional form over time, which is why a more complex approach of imputation is chosen. Therefore, we take the k-nearest neighbor method to impute the missing values based on all other columns excluding the timestamp column.
Resulting is the following imputation pattern:


| Column | Approach | 
|----------|------|
|DE_LU_price_day_ahead| k-Nearest Neighbors regression|
|DE_LU_solar_generation_actual| k-Nearest Neighbors regression|
|DE_LU_load_actual_entsoe_transparency| k-Nearest Neighbors regression|
|DE_wind_capacity| fill forward|
|DE_solar_capacity| fill forward|

In [25]:
# check which rows have missing values 
df[df[df.columns].isna().any(axis=1)]

,DE_LU_price_day_ahead,DE_LU_solar_generation_actual,DE_LU_wind_generation_actual,DE_LU_load_actual_entsoe_transparency,DE_wind_capacity,DE_solar_capacity,DE_temperature,DE_radiation_direct_horizontal,DE_radiation_diffuse_horizontal
cet_cest_timestamp,,,,,,,,,
2018-10-01 00:00:00+02:00,NaN,NaN,5932.0,NaN,47730.0,46099.0,8.676,0.0000,0.0000
2018-10-01 01:00:00+02:00,56.10,NaN,6042.0,NaN,47730.0,46099.0,8.258,0.0000,0.0000
2018-10-01 02:00:00+02:00,51.41,NaN,6021.0,41874.0,47730.0,46099.0,7.889,0.0000,0.0000
2018-10-01 03:00:00+02:00,47.38,NaN,6342.0,42713.0,47730.0,46099.0,7.620,0.0000,0.0000
2018-10-01 04:00:00+02:00,47.59,NaN,7144.0,44165.0,47730.0,46099.0,7.395,0.0000,0.0000
2018-10-01 05:00:00+02:00,51.61,NaN,7855.0,48435.0,47730.0,46099.0,7.146,0.0000,0.0000
2018-10-01 06:00:00+02:00,69.13,NaN,9493.0,57325.0,47730.0,46099.0,6.912,0.0000,0.0000
2019-01-03 09:00:00+01:00,60.03,1607.0,15654.0,NaN,48974.0,47565.0,-1.088,8.0657,53.1127
2019-01-03 10:00:00+01:00,58.19,3506.0,13401.0,NaN,48974.0,47565.0,0.021,23.8389,98.3175


In [32]:
# fill the missing values with the last valid value
df.sort_values("cet_cest_timestamp", inplace=True)
df[["DE_wind_capacity","DE_solar_capacity"]]= df[["DE_wind_capacity","DE_solar_capacity"]].ffill()

In [33]:
from sklearn.neighbors import KNeighborsRegressor

# impute the remaining missing values using KNN regression
impute_targets = [
    "DE_LU_load_actual_entsoe_transparency",
    "DE_LU_solar_generation_actual",
    "DE_LU_price_day_ahead",
]

predictor_cols = [
    "DE_LU_wind_generation_actual",
    "DE_wind_capacity",
    "DE_solar_capacity",
    "DE_temperature",
    "DE_radiation_direct_horizontal",
    "DE_radiation_diffuse_horizontal",
]

for target in impute_targets:
    valid_predictors = df[predictor_cols].notna().all(axis=1)
    train_mask = df[target].notna() & valid_predictors
    predict_mask = df[target].isna() & valid_predictors

    X_train = df.loc[train_mask, predictor_cols]
    y_train = df.loc[train_mask, target]
    X_pred = df.loc[predict_mask, predictor_cols]

    if len(X_train) == 0 or len(X_pred) == 0:
        continue

    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(X_train, y_train)
    df.loc[predict_mask, target] = knn.predict(X_pred)


In [34]:
# check for any remaining missing values
df.isna().sum()

DE_LU_price_day_ahead                    0
DE_LU_solar_generation_actual            0
DE_LU_wind_generation_actual             0
DE_LU_load_actual_entsoe_transparency    0
DE_wind_capacity                         0
DE_solar_capacity                        0
DE_temperature                           0
DE_radiation_direct_horizontal           0
DE_radiation_diffuse_horizontal          0
negative_price_flag                      0
dtype: int64

**3. Unequal distribution of positive and negative prices**


In [39]:
# check how many negative prices there are in the data -> class imbalance problem
df["negative_price_flag"] = (df["DE_LU_price_day_ahead"] < 0).astype(int)
display(df.groupby("negative_price_flag").count()["DE_LU_price_day_ahead"])

negative_price_flag
0    10732
1      238
Name: DE_LU_price_day_ahead, dtype: int64

Take away: A high class imbalance is apparent. This needs to be handled in the modeling part by balancing out the classes by using e.g. SMOTE or using a the correct evaluation metric (sensitivity instead of accuracy). Otherwise, the modell will classify all observations as positive.

## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [x] Causal graph / DAG (DoWhy)
- [ ] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

*Justification:* Gaining an deep understanding of the inference and causallity of the different variables.

### 4b. Supervised Learning
- [ ] Linear / Ridge / Lasso regression
- [ ] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [x] Decision Tree / Random Forest
- [x] Neural network (regression or classification)
- [ ] Other: ___

*Justification:* Using one simpler method as a baseline and comparing against the state of the art.

### 4c. Unsupervised Learning / Generative Models
- [x] K-Means clustering
- [ ] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

*Justification:* 


## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

Unsupervised Learning
- Using silhouette score to evaluate the clustering process

Supervised Learning
- Comparing the model against random classification
- Using the evaluation metric of sensivity of specificly identify negative prices
- Comparing Neural Network with Random Forrest



## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | | Data collection & cleaning |
| 2 | | Feature Construction |
| 3 | | Causal inference block |
| 4 | | Supervised learning block |
| 5 | | Unsupervised |
| 6 | | Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
